# LLM + Tools = A Simple Agent

This notebook introduces the smallest useful version of an agent:

```text
LLM decides what to do
tool performs a reliable operation
agent uses the tool result to answer
```

We compare:

1. **LLM-only** answering
2. **Agent with a tool**

The tool is intentionally simple: letter counting.

## Teaching frame

A careful way to introduce this:

> We have worked with LLMs as text generators and reasoners. But ChatGPT-style systems often do more than just generate text. They can use tools. A simple agent is an LLM wrapped in a decision process: it decides whether to call a tool, uses the result, and then answers.

This notebook focuses on one controlled pattern:

```text
LLM + tools + routing = simple agent
```

## Setup

Create a `.env` file in the same folder:

```text
OPENAI_API_KEY=your_key_here
```

Required packages:

```bash
pip install openai python-dotenv pydantic pandas
```

In [1]:
from dotenv import load_dotenv
from openai import OpenAI
from pydantic import BaseModel
from typing import Literal, Optional
import os
import json
import re
import pandas as pd

load_dotenv()

client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

MODEL = "gpt-4.1-mini"

## The tool

The tool does one thing reliably: count how many times a letter appears in text.

For exact symbolic tasks, a small deterministic tool can outperform an LLM.

In [2]:
def count_letters(text: str, letter: str) -> int:
    """Count occurrences of a letter in text. Case-insensitive."""
    return text.lower().count(letter.lower())


count_letters("strawberry", "r")

3

## Baseline: LLM-only answering

First, ask the LLM directly. No tool.

In [3]:
def llm_only_answer(question: str) -> str:
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": "Answer the user's question directly and briefly."},
            {"role": "user", "content": question},
        ],
        temperature=0,
    )

    return response.choices[0].message.content.strip()


question = "How many r's are in the word strawberry?"
print(llm_only_answer(question))

There are 2 r's in the word "strawberry."


## Define the planner output

The planner decides whether to answer directly or use the letter-counting tool.

In [4]:
class Plan(BaseModel):
    action: Literal["answer_directly", "use_count_letters"]
    text: Optional[str] = None
    letter: Optional[str] = None
    direct_answer: Optional[str] = None
    reason: str

## Planner

The LLM is now acting as a planner, not just as an answer generator.

In [5]:
def parse_json_safely(content: str) -> dict:
    """Parse JSON even if the model wraps it in a Markdown code fence."""
    content = content.strip()
    content = re.sub(r"^```json\s*", "", content)
    content = re.sub(r"^```\s*", "", content)
    content = re.sub(r"\s*```$", "", content)
    return json.loads(content)


def plan_next_action(question: str) -> Plan:
    prompt = f"""
You are a simple tool-using agent.

You have one tool available:

count_letters(text, letter)
- Use it when the user asks how many times a specific letter appears in a word or phrase.
- Do not count letters yourself when the tool applies.

If the question is not about counting letters, answer directly.

Return JSON only.

User question:
{question}

Return format:
{{
  "action": "use_count_letters" or "answer_directly",
  "text": "the word or phrase to inspect, or null",
  "letter": "the single letter to count, or null",
  "direct_answer": "answer if no tool is needed, otherwise null",
  "reason": "brief reason for the action"
}}
"""

    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": "You are a careful planner for a simple tool-using agent."},
            {"role": "user", "content": prompt},
        ],
        temperature=0,
    )

    content = response.choices[0].message.content
    data = parse_json_safely(content)
    return Plan(**data)

## Agent answer

Agent flow:

```text
question -> planner -> maybe call tool -> final answer
```

In [6]:
def agent_answer(question: str) -> dict:
    plan = plan_next_action(question)

    if plan.action == "answer_directly":
        return {
            "question": question,
            "path": "direct",
            "plan_reason": plan.reason,
            "answer": plan.direct_answer,
            "tool_result": None,
        }

    if plan.action == "use_count_letters":
        if not plan.text or not plan.letter:
            raise ValueError(f"Planner chose tool but omitted text or letter: {plan}")

        result = count_letters(plan.text, plan.letter)
        answer = f"The letter '{plan.letter}' appears {result} time(s) in '{plan.text}'."

        return {
            "question": question,
            "path": "tool",
            "plan_reason": plan.reason,
            "answer": answer,
            "tool_result": result,
        }

    raise ValueError(f"Unknown action: {plan.action}")

## Test both paths

These questions exercise both paths:

- letter-counting questions use the tool
- normal questions answer directly

In [7]:
test_questions = [
    "How many r's are in the word strawberry?",
    "How many s's are in Mississippi?",
    "What is a virtual environment in Python?",
    "Write a one-sentence tagline for a coffee shop.",
]

for q in test_questions:
    result = agent_answer(q)

    print("=" * 80)
    print("Question:", result["question"])
    print("Path:", result["path"])
    print("Planner reason:", result["plan_reason"])
    print("Answer:", result["answer"])

Question: How many r's are in the word strawberry?
Path: tool
Planner reason: The user asked how many times a specific letter appears in a word, so the counting tool should be used.
Answer: The letter 'r' appears 3 time(s) in 'strawberry'.
Question: How many s's are in Mississippi?
Path: tool
Planner reason: The user asked how many times a specific letter appears in a word, so the counting tool should be used.
Answer: The letter 's' appears 4 time(s) in 'Mississippi'.
Question: What is a virtual environment in Python?
Path: direct
Planner reason: The question is about explaining a concept, not counting letters.
Answer: A virtual environment in Python is an isolated environment that allows you to manage dependencies for different projects separately, preventing conflicts between packages and versions.
Question: Write a one-sentence tagline for a coffee shop.
Path: direct
Planner reason: The user asked for a tagline, which does not require counting letters.
Answer: Awaken your senses wit

## Side-by-side comparison

The point is not that the agent is always better.

The point is:

```text
For the right task, a tool makes the system more reliable.
```

In [8]:
comparison_questions = [
    "How many r's are in the word strawberry?",
    "How many s's are in Mississippi?",
    "How many e's are in the phrase never ever?",
    "What is a virtual environment in Python?",
]

rows = []

for q in comparison_questions:
    llm_answer = llm_only_answer(q)
    agent_result = agent_answer(q)

    rows.append({
        "question": q,
        "llm_only": llm_answer,
        "agent_path": agent_result["path"],
        "agent_answer": agent_result["answer"],
    })

pd.DataFrame(rows)

,question,llm_only,agent_path,agent_answer
0,How many r's are in the word strawberry?,"There are 2 r's in the word ""strawberry.""",tool,The letter 'r' appears 3 time(s) in 'strawberry'.
1,How many s's are in Mississippi?,"There are 4 s's in ""Mississippi.""",tool,The letter 's' appears 4 time(s) in 'Mississip...
2,How many e's are in the phrase never ever?,"There are 5 e's in the phrase ""never ever.""",tool,The letter 'e' appears 4 time(s) in 'never ever'.
3,What is a virtual environment in Python?,A virtual environment in Python is an isolated...,direct,A virtual environment in Python is an isolated...


## Visualize the agent flow

```mermaid
flowchart TD
    START --> planner
    planner -->|letter counting needed| count_letters_tool
    planner -->|no tool needed| direct_answer
    count_letters_tool --> final_answer
    direct_answer --> final_answer
    final_answer --> END
```

## Teaching checkpoint

Ask students:

1. Which part is the LLM?
2. Which part is the tool?
3. Which part makes this an agent instead of a plain LLM call?
4. Why is the tool more reliable for letter counting?
5. When would this pattern be overkill?

Suggested answer:

```text
This is agentic because the LLM decides between paths and may call a tool.
It is still simple and controlled. The agent does not have open-ended autonomy.
```

## What this is not

This is not a full autonomous agent.

A real-world agent may include:

- multiple tools
- multi-step loops
- memory/state
- retries
- human approval
- subgraphs

But the core idea starts here:

```text
LLM decides when to use a tool.
The tool performs work the LLM should not do from memory or token reasoning.
```